# **Medical RAG Chatbot Breast Cancer**



## 1. Install dependencies

In [1]:
!pip install -q faiss-cpu sentence-transformers openai scikit-learn pandas joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 82.7 MB/s eta 0:00:00


In [2]:
pip install python-dotenv

# 2. import labraries

In [3]:
import os
from pathlib import Path
from google.colab import files
import getpass
import json
import re
from openai import OpenAI
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import time
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

## 3. Upload your dataset



In [5]:
BASE_DIR = Path("/content")
KNOWLEDGE_DIR = BASE_DIR / "knowledge"
MODEL_DIR = BASE_DIR / "models"
ENV_DIR = BASE_DIR / "env"
KNOWLEDGE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
ENV_DIR.mkdir(exist_ok=True)

## 3. Auto-generate the knowledge base with an LLM


In [6]:
ENV_FILE = ENV_DIR / ".env"
base_url = "https://api.gapapi.com/v1"
model_name = "gpt-4o"

In [7]:
load_dotenv(ENV_FILE)
api_key = os.getenv("API_KEY")

In [8]:
os.environ["LLM_API_KEY"] = api_key

In [9]:
if api_key:
    os.environ["LLM_API_KEY"] = api_key
    client = OpenAI(api_key=api_key, base_url=base_url)
    use_llm = True
else:
    use_llm = False

In [10]:
KB_PATH = DATA_DIR / "knowledge_base.json"
MAX_RETRIES = 3
MIN_WORDS = 25
MAX_WORDS = 120

In [11]:
FEATURES = [
    ("vital status",            "patient outcome alive or dead at end of follow-up, the prediction target"),
    ("survival months",         "time from diagnosis to death or last contact in months; not a model input"),
    ("age at diagnosis",        "patient age in years at time of cancer diagnosis"),
    ("race",                    "race for disparity and subgroup analysis"),
    ("marital status",          "marital status at diagnosis, often used as a proxy for social support"),
    ("T stage",                 "primary tumor extent (T1 to T4) in the AJCC TNM system"),
    ("N stage",                 "regional lymph node involvement (N1 to N3) in the AJCC TNM system"),
    ("6th stage",               "overall AJCC 6th edition stage group (IIA to IIIC) derived from T, N and M"),
    ("tumor differentiation",   "how closely tumor cells resemble normal cells: well, moderately, poorly or undifferentiated"),
    ("tumor grade",             "histologic grade 1 to 3 reflecting cell differentiation"),
    ("A stage",                 "SEER summary stage: regional or distant spread"),
    ("tumor size",              "largest tumor dimension in millimeters at diagnosis"),
    ("ER status",               "estrogen receptor positivity from immunohistochemistry"),
    ("PR status",               "progesterone receptor positivity from immunohistochemistry"),
    ("regional nodes examined", "number of regional lymph nodes removed and examined by the pathologist"),
    ("regional nodes positive", "number of regional lymph nodes that contain tumor"),
    ("model limitations",       "SEER data gaps including missing chemotherapy, registry variation and censored follow-up"),
]


In [12]:
SYSTEM_PROMPT = (
    "You are an oncology data scientist writing concise reference entries about SEER "
    "(Surveillance, Epidemiology, and End Results) breast cancer registry data. "
    "For each topic write exactly one paragraph of three to four sentences. "
    "Cover the clinical meaning of the variable, how it is recorded in SEER, "
    "its role in survival or classification modeling, and any known data quality considerations. "
    "Write in plain prose with no markdown, bullet points, or headers. "
    "Do not exceed 120 words."
)


In [13]:
client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=base_url)

In [14]:
def _word_count(text):
    return len(re.findall(r"\w+", text))

In [15]:
def is_valid(text, topic):
    if not text:
        return False
    wc = _word_count(text)
    if wc < MIN_WORDS or wc > MAX_WORDS:
        return False
    topic_words = topic.lower().split()
    if not any(w in text.lower() for w in topic_words):
        return False
    lowered = text.lower()
    if any(p in lowered for p in ["i cannot", "i can't", "as an ai", "i'm sorry"]):
        return False
    return True

In [16]:
def generate_entry(idx, topic, hint):
    user_prompt = f"Topic: {topic}\nWhat it measures: {hint}"
    last_text = None
    for attempt in range(1, MAX_RETRIES + 1):
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.3,
        )
        text = response.choices[0].message.content.strip()
        last_text = text
        if is_valid(text, topic):
            return {"id": f"kb{idx:03d}", "topic": topic, "text": text}
        print(f"  [retry {attempt}/{MAX_RETRIES}] '{topic}' failed validation")
    print(f"  [warning] using unvalidated text for '{topic}'")
    return {"id": f"kb{idx:03d}", "topic": topic, "text": last_text, "needs_review": True}

In [17]:
if use_llm:
    kb = []
    for i, (topic, hint) in enumerate(FEATURES, start=1):
        print(f"Generating entry {i}/{len(FEATURES)}: {topic}")
        kb.append(generate_entry(i, topic, hint))

    with open(KB_PATH, "w", encoding="utf-8") as f:
        json.dump(kb, f, ensure_ascii=False, indent=2)

    flagged = [e for e in kb if e.get("needs_review")]
    print(f"\n{len(kb)} entries generated and saved to {KB_PATH}")
    if flagged:
        print(f"NOTE: {len(flagged)} entries need manual review (needs_review: true).")
else:
    raise SystemExit(
        "No API key available, so the knowledge base can't be auto-generated. "
        "Either provide a key above, or upload a pre-made knowledge_base.json to "
        f"{KB_PATH} yourself and skip this cell."
    )

Generating entry 1/17: vital status
Generating entry 2/17: survival months
Generating entry 3/17: age at diagnosis
Generating entry 4/17: race
Generating entry 5/17: marital status
Generating entry 6/17: T stage
Generating entry 7/17: N stage
Generating entry 8/17: 6th stage
Generating entry 9/17: tumor differentiation
Generating entry 10/17: tumor grade
Generating entry 11/17: A stage
Generating entry 12/17: tumor size
Generating entry 13/17: ER status
Generating entry 14/17: PR status
Generating entry 15/17: regional nodes examined
Generating entry 16/17: regional nodes positive
Generating entry 17/17: model limitations

17 entries generated and saved to /content/data/knowledge_base.json


In [18]:
with open(KB_PATH, "w", encoding="utf-8") as f:
    json.dump(kb, f, ensure_ascii=False, indent=2)

In [19]:
flagged = [e for e in kb if e.get("needs_review")]
print(f"{len(kb)} entries generated and saved to {KB_PATH}")
if flagged:
    print(f"{len(flagged)} flagged for review")

17 entries generated and saved to /content/data/knowledge_base.json


## 4. Build the FAISS retrieval index

In [20]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
KB_PATH = DATA_DIR / "knowledge_base.json"

In [21]:
with open(KB_PATH, "r", encoding="utf-8") as f:
    kb = json.load(f)

In [22]:
texts = [entry["text"] for entry in kb]
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")

Loading embedding model: all-MiniLM-L6-v2


In [23]:
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Embedding {len(texts)} knowledge-base passages...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 17 knowledge-base passages...


In [24]:
embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

In [25]:
faiss.write_index(index, str(MODEL_DIR / "kb.index"))
with open(MODEL_DIR / "kb_metadata.json", "w", encoding="utf-8") as f:
    json.dump(kb, f, ensure_ascii=False, indent=2)

## 5. Train the classifier

The dataset used in this project is sourced from SEER (Surveillance, Epidemiology, and End Results)—maintained by the U.S. National Cancer Institute—which is considered one of the world's most comprehensive cancer registries. A version of this dataset, widely used in machine learning projects, contains information on breast cancer patients characterized by 14 clinical features—such as tumor stage (T/N/6th Stage), hormone receptor status (ER/PR), tumor size, and the number of involved lymph nodes—with the objective of predicting patient survival or mortality (Alive/Dead). Among the research published using this dataset, the study most methodologically similar to the current project is **"An integrative machine learning framework for classifying SEER breast cancer dataset" (*Scientific Reports*, 2023)**, which compares various classification approaches—including Random Forest—applied to this specific data. In total, approximately **14,000** papers utilizing the SEER breast cancer dataset have been published in reputable sources, including prestigious journals such as **Nature**.

This dataset was modeled using **Random Forest** with **97%** accuracy.

In [26]:
model_path = MODEL_DIR / "classifier.joblib"

In [27]:
loaded = joblib.load(model_path)

In [28]:
DEFAULT_FEATURE_NAMES = [
    "Age", "Race", "Marital Status", "T Stage", "N Stage", "6th Stage",
    "differentiate", "Grade", "A Stage", "Tumor Size",
    "Estrogen Status", "Progesterone Status",
    "Regional Node Examined", "Reginol Node Positive",
]

In [29]:
DEFAULT_TARGET_NAMES = ["Alive", "Dead"]

In [30]:
if isinstance(loaded, dict) and "model" in loaded:
    clf = loaded["model"]
    feature_names = loaded.get("feature_names", DEFAULT_FEATURE_NAMES)
    target_names = loaded.get("target_names", DEFAULT_TARGET_NAMES)
else:
    clf = loaded
    feature_names = DEFAULT_FEATURE_NAMES
    target_names = DEFAULT_TARGET_NAMES

In [31]:
print("\nFeature names:", feature_names)


Feature names: ['Race', 'Marital Status', 'T Stage ', 'N Stage', '6th Stage', 'differentiate', 'Grade', 'A Stage', 'Estrogen Status', 'Progesterone Status', 'Age', 'Tumor Size', 'Regional Node Examined', 'Reginol Node Positive', 'Survival Months']


In [32]:
print("Target names:", target_names)

Target names: ['Alive', 'Dead']


In [33]:
ENCODING_MAPS = {
    "Race": {"White": 0, "Black": 1, "Other": 2},
    "Marital Status": {"Single": 0, "Married": 1, "Separated": 2, "Divorced": 3, "Widowed": 4},
    "T Stage": {"T1": 0, "T2": 1, "T3": 2, "T4": 3},
    "N Stage": {"N1": 0, "N2": 1, "N3": 2},
    "6th Stage": {"IIA": 0, "IIB": 1, "IIIA": 2, "IIIB": 3, "IIIC": 4},
    "differentiate": {
        "Well differentiated": 0,
        "Moderately differentiated": 1,
        "Poorly differentiated": 2,
        "Undifferentiated": 3,
    },
    "Grade": {"1": 0, "2": 1, "3": 2, "4": 3},
    "A Stage": {"Regional": 0, "Distant": 1},
    "Estrogen Status": {"Positive": 0, "Negative": 1},
    "Progesterone Status": {"Positive": 0, "Negative": 1},
}

In [34]:
NUMERIC_FEATURES = {"Age", "Tumor Size", "Regional Node Examined", "Reginol Node Positive","Survival Months"}

In [35]:
def encode_feature(name, raw_value):
    name = name.strip()
    if name in NUMERIC_FEATURES:
        try:
            return float(raw_value)
        except (ValueError, TypeError):
            raise ValueError(f"Feature '{name}' expects a numeric value, got: {repr(raw_value)}")
    mapping = ENCODING_MAPS.get(name, {})
    if not mapping:
        raise ValueError(f"Unknown feature: '{name}'")
    if raw_value not in mapping:
        raise ValueError(
            f"Value '{raw_value}' for '{name}' not recognized. "
            f"Valid options: {list(mapping.keys())}"
        )
    return mapping[raw_value]


## 6. Run the RAG chatbot

In [36]:
class MedicalRAGChatbot:
    def __init__(self, clf, feature_names, target_names, embedder, index, kb):
        self.clf = clf
        self.feature_names = feature_names
        self.target_names = target_names
        self.embedder = embedder
        self.index = index
        self.kb = kb
        self.use_llm = use_llm
        self.client = client if use_llm else None
        self.model_name = model_name

    def build_system_prompt(self, patient_values: dict, prediction: str, confidence: float) -> str:
        EXCLUDE_FIELDS = {"Survival Months"}
        lines = []
        for fname in self.feature_names:
            if fname in EXCLUDE_FIELDS:
                continue
            raw_val = patient_values.get(fname)
            if raw_val is None:
                continue
            encoded = int(raw_val)
            label_map = ENCODING_MAPS.get(fname)
            display = label_map[encoded] if label_map and encoded in label_map else str(raw_val)
            lines.append(f"  - {fname}: {display}")
        profile_block = "\n".join(lines)

        return (
            "You are a Clinical Decision Support (CDS) assistant specializing in breast cancer "
            "survival analysis. Your role is to help clinicians interpret ML-based survival "
            "predictions and provide evidence-based context — NOT to replace clinical judgment.\n\n"
            "## Patient Profile\n"
            f"{profile_block}\n\n"
            "## ML Model Output\n"
            f"  - Predicted Status: {prediction}\n"
            f"  - Model Confidence: {confidence:.1%}\n\n"
            "Guidelines:\n"
            "- Use ONLY the provided context passages to support your explanation.\n"
            "- Translate encoded values to clinically meaningful language.\n"
            "- Highlight risk factors present in this patient's profile.\n"
            "- Always recommend consultation with a qualified oncologist.\n"
            "- Do NOT mention Survival Months — it is not available at time of diagnosis."
        )

    def predict(self, encoded_values: list) -> dict:
        X = np.array([encoded_values])
        pred_idx = self.clf.predict(X)[0]
        proba = self.clf.predict_proba(X)[0]
        prediction = self.target_names[pred_idx]
        confidence = float(proba[pred_idx])
        return {"prediction": prediction, "confidence": confidence}

    def retrieve(self, query: str, k: int = 4) -> list:
        query_vec = self.embedder.encode([query], normalize_embeddings=True)
        scores, indices = self.index.search(query_vec, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < len(self.kb):
                entry = self.kb[idx]
                results.append({
                    "score": float(score),
                    "id": entry["id"],
                    "topic": entry["topic"],
                    "text": entry["text"],
                })
        return results

    def generate_explanation(
        self,
        user_question: str,
        prediction: str,
        confidence: float,
        passages: list,
        system_prompt_override: str = None,
    ) -> str:
        system_prompt = system_prompt_override or (
            "You are a medical education assistant. "
            "Use ONLY the provided context passages. "
            "Always remind the user to consult a qualified oncologist."
        )

        context_text = "\n\n".join(
            f"[{p['id']} | {p['topic']}]\n{p['text']}" for p in passages
        )
        user_prompt = (
            f"Question: {user_question}\n\n"
            f"Context Passages:\n{context_text}"
        )
        if self.use_llm:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
            )
            return response.choices[0].message.content.strip()
        else:
            return (
                f"Prediction: {prediction.upper()} (confidence: {confidence:.2%})\n\n"
                f"Relevant topics: {', '.join(p['topic'] for p in passages)}\n\n"
                "Note: LLM explanation unavailable."
            )

    def ask(self, encoded_values: list, patient_values: dict, user_question: str) -> str:
        result = self.predict(encoded_values)
        prediction = result["prediction"]
        confidence = result["confidence"]
        dynamic_system_prompt = self.build_system_prompt(patient_values, prediction, confidence)
        retrieval_query = f"{user_question} Prediction: {prediction}."
        passages = self.retrieve(retrieval_query)

        return self.generate_explanation(
            user_question, prediction, confidence, passages,
            system_prompt_override=dynamic_system_prompt,
        )

In [37]:
bot = MedicalRAGChatbot(clf, feature_names, target_names, embedder, index, kb)

## 7. Accurate cancer diagnosis through user questions


In [38]:
def bot_say(text, delay=3):
    print(f"TUMBot: {text}")
    if delay:
        time.sleep(delay)

In [39]:
bot_say("Welcome! I am TUMBot, your Clinical Decision Support assistant.")
bot_say("Please enter the following patient details:\n")

patient_values = {}

for fname in feature_names:
    while True:
        raw = input(f"  {fname}: ").strip()
        try:
            patient_values[fname] = encode_feature(fname, raw)
            break
        except ValueError as e:
            print(f"  [error] {e}. Please try again.")
print()
encoded_in_order = [patient_values[fname] for fname in feature_names]

result = bot.predict(encoded_in_order)
verdict = result["prediction"].upper()
conf_txt = f"{result['confidence']:.1%}"
bot_say("Got it , just a moment while the model runs...", delay=10)
print()
bot_say(f"Model prediction: {verdict}  ({conf_txt} confidence) , here's the reasoning:")

answer = bot.ask(encoded_in_order,patient_values,
    user_question="Why was this prediction made and what are the key risk factors?",)
print(answer)

bot_say(
    "Just a reminder: this tool is meant to support clinical decision-making, "
    "not replace it. Final diagnosis and treatment should always involve "
    "a qualified oncologist.",
    delay=0,
)


TUMBot: Welcome! I am TUMBot, your Clinical Decision Support assistant.
TUMBot: Please enter the following patient details:

  Race: Black
  Marital Status: 1
  [error] Value '1' for 'Marital Status' not recognized. Valid options: ['Single', 'Married', 'Separated', 'Divorced', 'Widowed']. Please try again.
  Marital Status: Single
  T Stage : T1
  N Stage: N2
  6th Stage: 1
  [error] Value '1' for '6th Stage' not recognized. Valid options: ['IIA', 'IIB', 'IIIA', 'IIIB', 'IIIC']. Please try again.
  6th Stage: IIIC
  differentiate: 1
  [error] Value '1' for 'differentiate' not recognized. Valid options: ['Well differentiated', 'Moderately differentiated', 'Poorly differentiated', 'Undifferentiated']. Please try again.
  differentiate: Poorly differentiated
  Grade: 6
  [error] Value '6' for 'Grade' not recognized. Valid options: ['1', '2', '3', '4']. Please try again.
  Grade: 4
  A Stage: 1
  [error] Value '1' for 'A Stage' not recognized. Valid options: ['Regional', 'Distant']. Please